In [5]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [6]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [7]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)
# it will generate a file named dataset.json with the generated dataset.


In [8]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

#This function processes every test case in our dataset and collects all the results into a single list.
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        print(f"Running test case: {test_case['task']}")
        result = run_test_case(test_case)
        results.append(result)
    
    return results

#Running the Evaluation
with open("dataset.json", "r") as f:
    dataset = json.load(f)

print("Running evaluation... with dataset:")
print(json.dumps(dataset, indent=2))
results = run_eval(dataset)

#Examining the Results
print("Evaluation results:")
print(json.dumps(results, indent=2))

Running evaluation... with dataset:
[
  {
    "task": "Write a Python function that extracts the AWS account ID from an ARN string. The function should take an ARN as input and return the 12-digit account ID."
  },
  {
    "task": "Create a JSON object that represents an IAM policy allowing read-only access to all objects in an S3 bucket named 'my-data-bucket'. Include the appropriate Action, Resource, and Effect fields."
  },
  {
    "task": "Write a regular expression that matches valid AWS S3 bucket names. S3 bucket names must be 3-63 characters long, contain only lowercase letters, numbers, hyphens, and periods, and cannot start or end with a hyphen or period."
  }
]
Running test case: Write a Python function that extracts the AWS account ID from an ARN string. The function should take an ARN as input and return the 12-digit account ID.
Running test case: Create a JSON object that represents an IAM policy allowing read-only access to all objects in an S3 bucket named 'my-data-bucke